In [ ]:
import numpy as np
import scanpy as sc
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import median_abs_deviation
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from scipy.sparse import csr_matrix, issparse

rcb.logger.setLevel(logging.ERROR)
sc.settings.verbosity = 0
sc.set_figure_params(dpi=80, facecolor='white', frameon=False)

%load_ext rpy2.ipython

# REMOVE them after finalizing
adata = sc.read_10x_h5('filtered_feature_bc_matrix.h5')
adata_raw = sc.read_10x_h5('raw_feature_bc_matrix.h5')

In [9]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
    np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

def filter_cells_by_mad(
    adata,
    mt_prefix,
    ribo_prefix,
    hb_prefix=r"^HB[^P]",
    top_gene_per=20,
    general_MAD=5,
    mt_MAD=3,
    mt_threshold=8
):
    '''
    Filter low-quality cells using Median Absolute Deviation (MAD).
    
    This function:
    - Calculates QC metrics for mitochondrial, ribosomal, and hemoglobin genes.
    - Detects outlier cells based on total counts, detected genes, and highly expressed genes.
    - Detects cells with abnormal or high mitochondrial content.
    - Removes the identified low-quality cells.
    - Returns the filtered AnnData object.
    '''   
    adata = adata.copy()
    adata.var_names_make_unique()
    
    # 1- QC stage
    adata.var["mt"] = adata.var_names.str.startswith(mt_prefix)
    adata.var["ribo"] = adata.var_names.str.startswith(ribo_prefix)
    adata.var["hb"] = adata.var_names.str.contains(hb_prefix,regex=True)

    sc.pp.calculate_qc_metrics(adata,
                               qc_vars=["mt", "ribo", "hb"],
                               percent_top=[top_gene_per],
                               log1p=True,
                               inplace=True)
    # 2- Detects outlier cells
    adata.obs['outlier'] = (is_outlier(adata, 'log1p_total_counts', general_MAD) |
                        is_outlier(adata, 'log1p_n_genes_by_counts', general_MAD) |
                        is_outlier(adata, f'pct_counts_in_top_{top_gene_per}_genes', general_MAD))

    # 3- Detects cells with abnormal or high mitochondrial content
    adata.obs['mt_outlier'] = is_outlier(adata, 'pct_counts_mt', mt_MAD)| (adata.obs["pct_counts_mt"] > mt_threshold)

    n_cells_before_filtering = adata.n_obs

    # 4- Filtering...
    adata = adata[(~adata.obs["outlier"]) & (~adata.obs["mt_outlier"])].copy()

    return adata, n_cells_before_filtering

def remove_ambient_rna_soupx(adata,
                     raw_matrix=None):
    '''
    Remove ambient RNA contamination using SoupX.

    This function:
    - Uses the raw and filtered count matrices.
    - Generates temporary cell clusters required by SoupX.
    - Estimates the ambient RNA contamination fraction.
    - Corrects gene expression counts.
    - Stores original and corrected counts in AnnData layers.
    - Returns the SoupX-corrected AnnData object.
    '''

    adata = adata.copy()
    
    adata_raw = raw_matrix.copy()
    adata_raw.var_names_make_unique()
    
    if not adata.var_names.equals(adata_raw.var_names):
        raise ValueError("Gene names/order differ between filtered and raw matrices.")
    
    data_tod = adata_raw.X.T
    tod_cells = adata_raw.obs_names
    
    del adata_raw
      
    SoupX = importr("SoupX")
    adata_pp = adata.copy()
    sc.pp.normalize_total(adata_pp)
    sc.pp.log1p(adata_pp)
    sc.pp.pca(adata_pp)
    sc.pp.neighbors(adata_pp)
    sc.tl.leiden(adata_pp,key_added="soupx_groups")
    soupx_groups = adata_pp.obs["soupx_groups"]

    del adata_pp
    
    cells = adata.obs_names
    genes = adata.var_names
    data = adata.X.T

    with localconverter(
        ro.default_converter +
        pandas2ri.converter +
        anndata2ri.converter
    ):
        ro.globalenv["data"] = data
        ro.globalenv["data_tod"] = data_tod
        ro.globalenv["genes"] = genes.to_numpy()
        ro.globalenv["cells"] = cells.to_numpy()
        ro.globalenv["tod_cells"] = tod_cells.to_numpy()
        ro.globalenv["soupx_groups"] = soupx_groups.astype(str)
        
    # Run SoupX in R
    ro.r("""
        rownames(data) <- genes
        colnames(data) <- cells
        
        rownames(data_tod) <- genes
        colnames(data_tod) <- tod_cells
        
        data <- as(data, "sparseMatrix")
        data_tod <- as(data_tod, "sparseMatrix")
        
        # Create SoupChannel and automatically estimate soup profile
        sc <- SoupChannel(
            data_tod,
            data,
            calcSoupProfile=TRUE
        )
        
        # Add cluster information
        sc <- setClusters(sc, soupx_groups)
        
        # Estimate contamination fraction
        sc <- autoEstCont(sc, doPlot=FALSE)
        
        # Correct counts
        out <- adjustCounts(sc, roundToInt=TRUE)
    """)
    with localconverter(
    ro.default_converter +
    pandas2ri.converter +
    anndata2ri.converter
    ):    
        out = ro.conversion.get_conversion().rpy2py(ro.globalenv["out"])

    # Store original counts
    adata.layers["counts"] = adata.X.copy()

    # Store SoupX-corrected counts
    adata.layers["soupx_counts"] = out.T
    
    # Continue downstream analysis using SoupX-corrected counts
    adata.X = adata.layers["soupx_counts"].copy()

    return adata

def remove_doublets(adata):
    '''
    Detect and remove doublets using scDblFinder.
    
    This function:
    - Converts the count matrix for use in R.
    - Detects doublets using scDblFinder.
    - Stores the doublet score and classification for each cell.
    - Removes cells classified as doublets.
    - Returns the filtered AnnData object and the number of cells before doublet removal.
    '''
    adata = adata.copy()
        
    Seurat = importr("Seurat")
    scater = importr("scater")
    scDblFinder = importr("scDblFinder")
    BiocParallel = importr("BiocParallel")
    
    data_mat = adata.X.T
        
    with localconverter(
        ro.default_converter +
        pandas2ri.converter +
        anndata2ri.converter
    ):
        ro.globalenv["data_mat"] = data_mat

    ro.r("""
        set.seed(123)

        sce <- SingleCellExperiment(list(counts=data_mat))
        sce <- scDblFinder(sce)

        doublet_score <- sce$scDblFinder.score
        doublet_class <- as.character(sce$scDblFinder.class)
        
    """)

    doublet_score = np.asarray(ro.globalenv["doublet_score"])
    doublet_class = np.asarray(ro.globalenv["doublet_class"])

    adata.obs['doublet_score'] = doublet_score
    adata.obs['doublet_class'] = doublet_class

    # Remove detected doublets
    n_cells_before_doublet_filtering = adata.n_obs
    adata = adata[
        adata.obs['doublet_class']=='singlet'
        ].copy()
    
    return adata, n_cells_before_doublet_filtering

def filter_genes_by_min_cells(adata, min_cells):
    '''
    Filter genes based on the minimum number of cells.

    This function:
    - Removes genes detected in fewer than `min_cells` cells.
    - Removes low-information genes from downstream analysis.
    - Returns the filtered AnnData object and the number of genes before filtering.
    '''
    
    adata = adata.copy()
    n_genes_before_min_cells_removing = adata.n_vars
    sc.pp.filter_genes(adata,min_cells=min_cells)

    return adata, n_genes_before_min_cells_removing

def shifted_normalization(adata):
    '''
    Normalize gene expression using library-size scaling followed by log1p transformation.
    
    This function:
    - Normalizes counts to account for differences in sequencing depth between cells.
    - Applies a log1p transformation to reduce the influence of highly expressed genes.
    - Stores the normalized values in the `log1p_norm` layer.
    - Returns the AnnData object with the normalized layer.
    '''
    scales_counts = sc.pp.normalize_total(adata,target_sum=None,inplace=False)
    adata.layers['log1p_norm'] = sc.pp.log1p(scales_counts['X'],copy=True)
    
    return adata

def scran_normalization(adata):  
    '''
    Normalize gene expression using scran size factors.
    
    This function:
    - Performs temporary preprocessing and clustering of cells.
    - Uses scran to estimate cell specific size factors based on pooled counts.
    - Corrects counts for differences in sequencing depth and composition.
    - Applies a log1p transformation to the normalized counts.
    - Stores the size factors in `adata.obs`.
    - Stores normalized values in the `scran_normalization` layer.
    - Returns the AnnData object with the normalized layer.
    '''
    scran = importr("scran")
    BiocParallel = importr("BiocParallel")

    adata_pp = adata.copy()
    sc.pp.normalize_total(adata_pp)
    sc.pp.log1p(adata_pp)
    sc.pp.pca(adata_pp, n_comps=15)
    sc.pp.neighbors(adata_pp)
    sc.tl.leiden(adata_pp,key_added="group")

    data_mat = adata.X.T
    if issparse(data_mat):
        if data_mat.nnz>2**31-1:
            data_mat = data_mat.tocoo()
        else:
            data_mat = data_mat.tocsc()

    with localconverter(
    ro.default_converter +
    pandas2ri.converter +
    anndata2ri.converter
    ):
        ro.globalenv["data_mat"] = data_mat
        ro.globalenv["input_groups"] = adata_pp.obs["group"]
        del adata_pp
    
    ro.r('''size_factors = sizeFactors(
    computeSumFactors(
        SingleCellExperiment(
            list(counts=data_mat)),
            clusters = input_groups,
            min.mean = 0.1,
            BPPARAM = MulticoreParam()
            )
        )''')
    size_factors = np.asarray(ro.globalenv["size_factors"])
    adata.obs['size_factors'] = size_factors
    
    scran_norm = adata.X/adata.obs['size_factors'].values[:,None]
    scran_norm = csr_matrix(scran_norm)
    scran_norm.data = np.log1p(scran_norm.data)
    adata.layers["scran_normalization"] = scran_norm

    return adata

def pearson_normalization(adata): 
    '''
    Normalize gene expression using analytic Pearson residuals.
    
    This function:
    - Models expected gene counts while accounting for differences in sequencing depth.
    - Calculates Pearson residuals representing deviations from expected expression.
    - Reduces technical variation while preserving biologically informative variation.
    - Stores the normalized values in the `analytic pearson residuals` layer.
    - Returns the AnnData object with the normalized layer.
    '''
    analytic_pearson = sc.experimental.pp.normalize_pearson_residuals(adata,inplace=False)
    adata.layers["analytic_pearson_residuals"] = csr_matrix(analytic_pearson['X'])
    
    return adata

def feature_selection(adata):
    

In [12]:
def run_pipeline(adata,
                     raw_matrix,
                     mt_prefix="MT-",
                     ribo_prefix=("RPS", "RPL"),
                     hb_prefix=r"^HB[^P]",
                     top_gene_per=20,
                     general_MAD=5,
                     mt_MAD=3,
                     mt_threshold=8,
                     min_cells=20,
                 normalization_method="log1p"
                ):
    
    # 1. QC filtering
    adata, n_cells_before_filtering = filter_cells_by_mad(adata,
                                                           mt_prefix,
                                                           ribo_prefix,
                                                           hb_prefix,
                                                           top_gene_per,
                                                           general_MAD,
                                                           mt_MAD,
                                                           mt_threshold
                                                         )
    '''
    Run the complete scRNA-seq preprocessing pipeline.
    
    This function:
    - Filters low-quality cells using MAD-based QC thresholds.
    - Corrects ambient RNA contamination using SoupX.
    - Detects and removes doublets using scDblFinder.
    - Removes genes detected in fewer than the specified number of cells.
    - Normalizes gene expression using the selected normalization method.
    - Returns the processed AnnData object and preprocessing statistics.
    
    Parameters:
    - adata: Filtered count matrix as an AnnData object.
    - raw_matrix: Raw count matrix required for SoupX.
    - mt_prefix: Prefix used to identify mitochondrial genes.
    - ribo_prefix: Prefixes used to identify ribosomal genes.
    - hb_prefix: Pattern used to identify hemoglobin genes.
    - top_gene_per: Number of top-expressed genes used for QC.
    - general_MAD: MAD threshold for general QC outlier detection.
    - mt_MAD: MAD threshold for mitochondrial outlier detection.
    - mt_threshold: Maximum allowed mitochondrial percentage.
    - min_cells: Minimum number of cells in which a gene must be detected.
    - normalization_method: Normalization method to use:
      "log1p", "scran", or "pearson".
    '''
    
    # 2. Ambient RNA correction
    adata = remove_ambient_rna_soupx(adata, raw_matrix)
    
    # 3. Doublet removal
    adata, n_cells_before_doublet = remove_doublets(adata)
    
    # 4. Gene filtering
    adata, n_genes_before_filtering = filter_genes_by_min_cells(adata,min_cells)

    # 5. Normalization
    if normalization_method =='log1p':
        adata = shifted_normalization(adata)
    elif normalization_method == 'scran':
        adata = shifted_normalization(adata)
    elif normalization_method == 'pearson':
        adata = pearson_normalization(adata)
    else:
        raise ValueError(
        f"Invalid normalization method: '{normalization_method}'. "
        "Choose 'log1p', 'scran', or 'pearson'."
    )
    
    return (
        adata,
        n_cells_before_filtering,
        n_cells_before_doublet,
        n_genes_before_filtering
    )

In [13]:
output = run_pipeline(adata,adata_raw)

/home/farshid/miniconda/envs/pyroe_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/farshid/miniconda/envs/pyroe_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [16]:
output

(AnnData object with n_obs × n_vars = 2438 × 13477
     obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'doublet_score', 'doublet_class'
     var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
     layers: 'counts', 'soupx_counts', 'log1p_norm',
 4804,
 2593,
 36601)